In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.onnx

In [ ]:
train_images, train_labels = torch.load("/content/drive/MyDrive/data/train_data.pt")
test_images, test_labels = torch.load("/content/drive/MyDrive/data/test_data.pt")

In [ ]:
train_dataset = TensorDataset(train_images, train_labels)
test_dataset = TensorDataset(test_images, test_labels)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Pooling
        self.pool = nn.MaxPool2d(2, 2)

        # Fully connected layers
        self.fc1 = nn.Linear(64 * 28 * 28, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)  # flatten

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN().to(device)

print(model)

SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=50176, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
)


In [ ]:
criterion = nn.CrossEntropyLoss()   # for classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {running_loss:.4f}")

Epoch [1/10], Loss: 15.1387
Epoch [2/10], Loss: 12.3311
Epoch [3/10], Loss: 11.9038
Epoch [4/10], Loss: 11.4391
Epoch [5/10], Loss: 10.5939
Epoch [6/10], Loss: 9.0971
Epoch [7/10], Loss: 7.8123
Epoch [8/10], Loss: 6.3744
Epoch [9/10], Loss: 4.0738
Epoch [10/10], Loss: 2.5410


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 65.71%


In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/data/cnn_model.pth")

In [ ]:
import os
!pip install onnxscript

# Load the trained model
model = SimpleCNN().to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/data/cnn_model.pth", weights_only=False))
model.eval()

dummy_input = torch.randn(1, 3, 224, 224).to(device)

# Export the model to ONNX format
torch.onnx.export(
    model,
    dummy_input,
    "/content/drive/MyDrive/data/cnn_model.onnx",
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names = ['input'],
    output_names = ['output']
)

print("Model exported to cnn_model.onnx successfully!")

In [ ]:
import onnx
import os

# Define the paths
onnx_model_path = "/content/drive/MyDrive/data/cnn_model.onnx"
merged_onnx_model_path = "/content/drive/MyDrive/data/cnn_model_merged.onnx"

# Load the ONNX model
try:
    model = onnx.load(onnx_model_path)
    print(f"Successfully loaded ONNX model from {onnx_model_path}")

    # Save the model back as a single file, disabling external data storage
    onnx.save(model, merged_onnx_model_path, save_as_external_data=False)
    print(f"Successfully merged and saved ONNX model to {merged_onnx_model_path} as a single file.")

except Exception as e:
    print(f"An error occurred during ONNX model merging: {e}")